# House Price Prediction Using Machine Learning

## Project Overview

This project develops and evaluates machine learning models to predict residential property prices using the Kaggle House Prices dataset.

The project explores data preprocessing, feature engineering, regression models, and model evaluation to improve prediction accuracy.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings("ignore")

In [2]:
train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)


# Target & IDs

In [3]:
y = np.log1p(train["SalePrice"])
test_ids = test["Id"]

# Create fresh copies from the original datasets
train_features = train.drop(columns=["SalePrice", "Id"]).copy()
test_features = test.drop(columns=["Id"]).copy()

# Feature Engineering

In [4]:

def add_features(df):
    df = df.copy()

    # Total Square Footage
    df["TotalSF"] = (
        df["TotalBsmtSF"].fillna(0)
        + df["1stFlrSF"].fillna(0)
        + df["2ndFlrSF"].fillna(0)
    )

    # Total Bathrooms
    df["TotalBath"] = (
        df["FullBath"].fillna(0)
        + 0.5 * df["HalfBath"].fillna(0)
        + df["BsmtFullBath"].fillna(0)
        + 0.5 * df["BsmtHalfBath"].fillna(0)
    )

    # House and renovation age
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]

    # Garage age
    df["GarageAge"] = df["YrSold"] - df["GarageYrBlt"].fillna(df["YearBuilt"])

    # Interaction features
    df["GrLiv_Overall"] = df["GrLivArea"] * df["OverallQual"]
    df["TotalSF_Overall"] = df["TotalSF"] * df["OverallQual"]

    # Remove redundant columns
    df = df.drop(
        columns=["1stFlrSF", "2ndFlrSF", "TotalBsmtSF"]
    )

    return df


# Apply feature engineering separately
train_features = add_features(train_features)
test_features = add_features(test_features)

print("Training features:", train_features.shape)
print("Test features:", test_features.shape)


Training features: (1460, 83)
Test features: (1459, 83)


# Data Cleaning

In [5]:

# Identify numeric and categorical columns
num_cols = train_features.select_dtypes(include=["number"]).columns
cat_cols = train_features.select_dtypes(include=["object"]).columns

# Learn median values from training data only
train_medians = train_features[num_cols].median()

# Create separate copies
train_clean = train_features.copy()
test_clean = test_features.copy()

# Fill missing numeric values
train_clean[num_cols] = train_clean[num_cols].fillna(train_medians)
test_clean[num_cols] = test_clean[num_cols].fillna(train_medians)

# Fill missing categorical values
train_clean[cat_cols] = train_clean[cat_cols].fillna("None")
test_clean[cat_cols] = test_clean[cat_cols].fillna("None")

# Convert categorical features into numeric columns
X = pd.get_dummies(train_clean, drop_first=True)
X_test = pd.get_dummies(test_clean, drop_first=True)

# Ensure both datasets have the same columns
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print("Training shape:", X.shape)
print("Test shape:", X_test.shape)


Training shape: (1460, 264)
Test shape: (1459, 264)


In [6]:
from sklearn.model_selection import cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=kf,
        scoring="neg_root_mean_squared_error"
    )

    rmse = -scores.mean()
    results.append({"Model": name, "CV RMSE": rmse})
    print(f"{name}: {rmse:.4f}")

results_df = pd.DataFrame(results).sort_values("CV RMSE")
display(results_df)


Linear Regression: 0.1645
Random Forest: 0.1443
XGBoost: 0.1325


,Model,CV RMSE
2,XGBoost,0.132477
1,Random Forest,0.144345
0,Linear Regression,0.164521


### Initial Model Evaluation Results

Three regression models were evaluated using 5-fold cross-validation.

- Linear Regression: RMSE = 0.1645
- Random Forest: RMSE = 0.1443
- XGBoost: RMSE = 0.1325

XGBoost achieved the lowest cross-validation RMSE, suggesting that it captured the relationships between housing features and sale prices more effectively than the other two models.

These preliminary results use preprocessing fitted before cross-validation. A leakage-safe evaluation will be performed before final model selection.

In [7]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Use the feature-engineered data before cleaning
numeric_cols = train_features.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = train_features.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Prepare numeric and categorical columns separately
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(
                strategy="constant", fill_value="None"
            )),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore"
            ))
        ]), categorical_cols)
    ]
)

# Start with XGBoost, our best preliminary model
xgb_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        random_state=42,
        n_jobs=-1
    ))
])

# Evaluate using leakage-safe cross-validation
safe_scores = cross_val_score(
    xgb_pipeline,
    train_features,
    y,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_root_mean_squared_error"
)

print("Leakage-safe XGBoost RMSE:", round(-safe_scores.mean(), 4))


Leakage-safe XGBoost RMSE: 0.1305


### Leakage-Safe Model Evaluation

To improve the reliability of model evaluation, a Scikit-learn Pipeline was implemented.

The pipeline applies missing-value imputation and categorical encoding separately within each cross-validation fold, preventing information leakage from validation data.

Using 5-fold cross-validation, the XGBoost model achieved an RMSE of **0.1305** on log-transformed house prices.

This result provides a more reliable estimate of the model's predictive performance than the initial evaluation.

In [8]:
# Compare all three models using leakage-safe pipelines
safe_results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        train_features,
        y,
        cv=KFold(n_splits=5, shuffle=True, random_state=42),
        scoring="neg_root_mean_squared_error"
    )

    rmse = -scores.mean()

    safe_results.append({
        "Model": name,
        "CV RMSE": rmse
    })

    print(f"{name}: {rmse:.4f}")

# Display results from best to worst
safe_results_df = pd.DataFrame(safe_results)
safe_results_df = safe_results_df.sort_values("CV RMSE")

display(safe_results_df)


Linear Regression: 0.1469
Random Forest: 0.1430
XGBoost: 0.1305


,Model,CV RMSE
2,XGBoost,0.130522
1,Random Forest,0.142979
0,Linear Regression,0.146894


### Final Model Comparison

Three regression models were evaluated using 5-fold cross-validation with leakage-safe preprocessing.

| Model | CV RMSE |
|---|---|
| XGBoost | 0.1305 |
| Random Forest | 0.1430 |
| Linear Regression | 0.1469 |

**XGBoost achieved the lowest cross-validation RMSE (0.1305)** and was selected as the best-performing model among the three evaluated approaches.

All scores were calculated on log-transformed house prices. Lower RMSE indicates better predictive performance.

In [9]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler

# Models used in stacking
base_models = [
    ("rf", RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    )),
    ("xgb", XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        random_state=42,
        n_jobs=-1
    )),
    
("ridge", Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("model", Ridge(alpha=1.0))
])),
("lasso", Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("model", Lasso(alpha=0.001, max_iter=10000))
]))

]

print("Stacking base models prepared!")


Stacking base models prepared!


In [10]:
# Train the selected pipeline on all training houses
xgb_pipeline.fit(train_features, y)

# Predict log-transformed prices for test houses
test_predictions_log = xgb_pipeline.predict(test_features)

# Convert predictions back to actual house prices
test_predictions = np.expm1(test_predictions_log)

# Create Kaggle submission file
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_predictions
})

# Save the predictions
submission.to_csv("submission.csv", index=False)

# Preview the results
print("Submission shape:", submission.shape)
display(submission.head())


Submission shape: (1459, 2)


,Id,SalePrice
0,1461,121270.679688
1,1462,165716.171875
2,1463,172292.437500
3,1464,187003.750000
4,1465,184170.015625


In [11]:
from sklearn.base import clone

# Give each model its own preprocessing pipeline
stack_estimators = []

for name, model in base_models:
    stack_estimators.append((
        name,
        Pipeline([
            ("preprocessing", clone(preprocessor)),
            ("model", clone(model))
        ])
    ))

# Combine the four models using Ridge as the meta-model
stacking_model = StackingRegressor(
    estimators=stack_estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

# Evaluate the entire stacking model
stack_scores = cross_val_score(
    stacking_model,
    train_features,
    y,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_root_mean_squared_error",
    n_jobs=1
)

stack_rmse = -stack_scores.mean()

print("Stacking CV RMSE:", round(stack_rmse, 4))
print("XGBoost CV RMSE:", 0.1305)


Stacking CV RMSE: 0.1308
XGBoost CV RMSE: 0.1305


### Stacking Ensemble Experiment

To investigate whether combining multiple models could improve prediction accuracy, a stacking ensemble was developed using Random Forest, XGBoost, Ridge, and Lasso, with Ridge as the final meta-model.

Preprocessing and feature scaling were incorporated into the model pipelines to prevent data leakage during cross-validation.

**Results (5-fold cross-validation):**

| Approach | CV RMSE |
|---|---|
| XGBoost | 0.1305 |
| Stacking Ensemble | 0.1308 |

The stacking ensemble achieved performance comparable to XGBoost but did not improve the average validation score.

Therefore, **XGBoost was selected as the final model**, offering similar predictive performance with lower complexity.